In [1]:
# ==========================================
# AI RECRUITMENT MODEL TRAINING
# Google Colab Version
# ==========================================

!pip install xgboost -q

import os
import joblib
import pandas as pd
import numpy as np

from google.colab import files

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

# ==========================================
# UPLOAD DATASET
# ==========================================

uploaded = files.upload()

DATASET_PATH = list(uploaded.keys())[0]

MODELS_DIR = "models"

os.makedirs(MODELS_DIR, exist_ok=True)

# ==========================================
# LOAD DATASET
# ==========================================

print("Loading dataset...")

df = pd.read_csv(DATASET_PATH)

print("\nDataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

# ==========================================
# FEATURES & TARGET
# ==========================================

X = df.drop(columns=["HiringDecision"])

y = df["HiringDecision"]

# ==========================================
# FEATURE GROUPS
# ==========================================

numeric_features = [
    "Age",
    "DistanceFromCompany",
    "InterviewScore",
    "SkillScore",
    "PersonalityScore",
    "ExperienceYears",
    "PreviousCompanies"
]

categorical_features = [
    "RecruitmentStrategy"
]

passthrough_features = [
    "Gender",
    "EducationLevel"
]

# ==========================================
# PREPROCESSING
# ==========================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "pass",
            "passthrough",
            passthrough_features
        )
    ]
)

# ==========================================
# SPLIT DATA
# ==========================================

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

print(f"\nTrain Size : {len(X_train)}")
print(f"Test Size  : {len(X_test)}")

# ==========================================
# MODELS
# ==========================================

models = {

    "Logistic Regression":
        LogisticRegression(
            max_iter=1000,
            random_state=42
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingClassifier(
            n_estimators=100,
            random_state=42
        ),

    "XGBoost":
        XGBClassifier(
            random_state=42,
            eval_metric="logloss"
        )
}

# ==========================================
# CROSS VALIDATION
# ==========================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = {}

print("\n")
print("="*60)
print("MODEL COMPARISON")
print("="*60)

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=[
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc"
        ]
    )

    results[name] = {
        "accuracy":
            np.mean(scores["test_accuracy"]),
        "precision":
            np.mean(scores["test_precision"]),
        "recall":
            np.mean(scores["test_recall"]),
        "f1":
            np.mean(scores["test_f1"]),
        "roc_auc":
            np.mean(scores["test_roc_auc"])
    }

    print(f"\n{name}")
    print("-"*30)

    print(
        f"Accuracy  : {results[name]['accuracy']:.4f}"
    )

    print(
        f"Precision : {results[name]['precision']:.4f}"
    )

    print(
        f"Recall    : {results[name]['recall']:.4f}"
    )

    print(
        f"F1 Score  : {results[name]['f1']:.4f}"
    )

    print(
        f"ROC AUC   : {results[name]['roc_auc']:.4f}"
    )

# ==========================================
# BEST MODEL
# ==========================================

best_model_name = max(
    results,
    key=lambda x:
    results[x]["roc_auc"]
)

print("\n")
print("="*60)
print("BEST MODEL")
print("="*60)

print(best_model_name)

best_model = models[best_model_name]

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", best_model)
])

# ==========================================
# FINAL TRAINING
# ==========================================

pipeline.fit(X_train, y_train)

preds = pipeline.predict(X_test)

probs = pipeline.predict_proba(X_test)[:,1]

print("\n")
print("="*60)
print("FINAL EVALUATION")
print("="*60)

print(
    f"Accuracy : {accuracy_score(y_test,preds):.4f}"
)

print(
    f"Precision: {precision_score(y_test,preds):.4f}"
)

print(
    f"Recall   : {recall_score(y_test,preds):.4f}"
)

print(
    f"F1 Score : {f1_score(y_test,preds):.4f}"
)

print(
    f"ROC AUC  : {roc_auc_score(y_test,probs):.4f}"
)

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        preds
    )
)

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        y_test,
        preds
    )
)

# ==========================================
# RETRAIN ALL DATA
# ==========================================

print("\nRetraining on full dataset...")

pipeline.fit(X, y)

# ==========================================
# SAVE MODEL
# ==========================================

joblib.dump(
    pipeline,
    "recruitment_model.pkl"
)

print("\nModel saved:")
print("recruitment_model.pkl")

# ==========================================
# DOWNLOAD MODEL
# ==========================================

files.download(
    "recruitment_model.pkl"
)

Saving recruitment_data.csv to recruitment_data.csv
Loading dataset...

Dataset Shape:
(1500, 11)

Columns:
['Age', 'Gender', 'EducationLevel', 'ExperienceYears', 'PreviousCompanies', 'DistanceFromCompany', 'InterviewScore', 'SkillScore', 'PersonalityScore', 'RecruitmentStrategy', 'HiringDecision']

Train Size : 1200
Test Size  : 300


MODEL COMPARISON

Logistic Regression
------------------------------
Accuracy  : 0.8725
Precision : 0.8277
Recall    : 0.7475
F1 Score  : 0.7848
ROC AUC   : 0.9101

Random Forest
------------------------------
Accuracy  : 0.8992
Precision : 0.8929
Recall    : 0.7663
F1 Score  : 0.8245
ROC AUC   : 0.9220

Gradient Boosting
------------------------------
Accuracy  : 0.9217
Precision : 0.9069
Recall    : 0.8333
F1 Score  : 0.8681
ROC AUC   : 0.9327

XGBoost
------------------------------
Accuracy  : 0.9283
Precision : 0.9117
Recall    : 0.8521
F1 Score  : 0.8805
ROC AUC   : 0.9336


BEST MODEL
XGBoost


FINAL EVALUATION
Accuracy : 0.9233
Precision: 0.9268
R

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>